# Enzyme Representation x F_physchem_onehot: Full Combination Analysis

The amine representation comparison found **F_physchem_onehot** (physicochemical + one-hot, 41 dims) as the best amine representation, but only tested one enzyme representation (noncons_mean, conservation < 0.5). The nonconserved residue notebook tested many enzyme representations but only with Morgan 1024 fingerprints.

**This notebook tests the best amine representation against all 8 enzyme representations**, and also analyzes the splits themselves to understand what drives performance variation.

| Code | Representation | Dims |
|------|---------------|------|
| full_protein | Mean of all residues | 1024 |
| very_unique | Conservation < 0.3, mean pool | 1024 |
| unique | Conservation < 0.5, mean pool | 1024 |
| non_conserved | Conservation < 0.8, mean pool | 1024 |
| all_variable | Conservation < 0.95, mean pool | 1024 |
| noncons_max | Conservation < 0.5, max pool | 1024 |
| noncons_mean_max | Conservation < 0.5, mean+max concat | 2048 |
| cons_plus_noncons | Conserved(mean) + non-conserved(mean) | 2048 |

## 1. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import h5py
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score, log_loss
)

import xgboost as xgb

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors

from Bio import SeqIO

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")

COMBO_DIR = OUTPUT_DIR / "model_outputs" / "enzyme_amine_combination"
COMBO_DIR.mkdir(exist_ok=True, parents=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

N_SPLITS = 10
SEEDS = [42, 123, 456, 789, 1011, 2022, 3033, 4044, 5055, 6066]

print("Setup complete.")
print(f"Output directory: {COMBO_DIR}")

## 2. Load Data

In [ ]:
# --- Per-residue ProtT5 embeddings ---
h5_path = DATA_DIR / "Seqs_list_total_per_residue.h5"
per_residue_embeddings = {}
with h5py.File(h5_path, 'r') as f:
    for key in f.keys():
        uniprot_id = key.split('_')[-1]
        per_residue_embeddings[uniprot_id] = f[key][:]
print(f"Per-residue embeddings: {len(per_residue_embeddings)} enzymes")

# --- Full protein embeddings (mean-pooled) ---
h5_full = DATA_DIR / "Seqs_list_total.h5"
full_embeddings = {}
with h5py.File(h5_full, 'r') as f:
    for key in f.keys():
        uniprot_id = key.split('_')[-1]
        full_embeddings[uniprot_id] = f[key][:]
print(f"Full protein embeddings: {len(full_embeddings)} enzymes")

# --- Conservation scores ---
df_cons = pd.read_csv(OUTPUT_DIR / "conservation_scores.csv")
core_mask = df_cons['gap_fraction'] < 0.5
df_core = df_cons[core_mask].copy()
print(f"Core alignment positions (gap < 50%): {len(df_core)}")

# --- Alignment mapping ---
alignment_to_seq = {}
for record in SeqIO.parse(OUTPUT_DIR / "bsh_aligned.fasta", 'fasta'):
    parts = record.id.split('_')
    uniprot_id = parts[-1] if len(parts) > 1 else record.id
    seq = str(record.seq)
    mapping = {}
    seq_pos = 0
    for aln_pos, char in enumerate(seq):
        if char != '-':
            mapping[aln_pos] = seq_pos
            seq_pos += 1
    alignment_to_seq[uniprot_id] = mapping
overlap = set(alignment_to_seq.keys()) & set(per_residue_embeddings.keys())
print(f"Enzymes with alignment + embeddings: {len(overlap)}")

# --- Activity labels ---
df_activity = pd.read_csv(OUTPUT_DIR / "enzyme_amine_activity.csv")
controls = ['CTRL1', 'CTRL2', 'CTRL3', 'CTRL4', 'CTRL5', 'CTRL6', 'CTRL7']
canonical = ['taurine', 'glycine']
df_activity = df_activity[~df_activity['Enzyme'].isin(controls)]
df_activity = df_activity[~df_activity['Amine'].isin(canonical)]
df_agg = df_activity.groupby(['Enzyme', 'Amine']).agg(
    active=('active_approach2', 'any'),
    n_products=('ProductName', 'count'),
    n_active_products=('active_approach2', 'sum')
).reset_index()
print(f"Activity data: {df_agg.shape[0]} enzyme-amine pairs")
print(f"  Active: {df_agg['active'].sum()}, Inactive: {(~df_agg['active']).sum()} ({df_agg['active'].mean():.1%} active)")

# --- Amine SMILES ---
df_smiles = pd.read_excel(DATA_DIR / "bsh_reactants_SMILES_corrected.xlsx")

name_map = {
    '2,3-Diaminopropinoic Acid': '2,3_diaminopropionic acid',
    '2-aminophenol': '2_aminophenol',
    '3-methoxytyramine HCl': '3_methoxytyramine',
    '4-aminophenol': '4_aminophenol',
    'L-Alanine': 'alanine',
    'L-Arginine': 'arginine',
    'Asparagine': 'asparagine',
    'Cadaverine': 'cadaverine',
    'L-Citrulline': 'citrulline',
    'L-Cysteine': 'cysteine',
    'Dopamine HCl': 'dopamine',
    'gamma-Aminobutyric acid >99%': 'gaba',
    'L-Glutamine': 'glutamine',
    'Glycyl-L-Valine': 'glyglycine',
    'L-Histidine': 'histidine',
    'L-Lysine': 'lysine',
    'L-Methionine': 'methionine',
    'L-Ornithine monohydrochloride': 'ornithine',
    'L-Phenylalanine': 'phenylalanine',
    'DL-Proline': 'proline',
    'Putrescine': 'putrescine',
    'L-Serine': 'serine',
    'L-Threonine': 'threonine',
    'Tryptamine': 'tryptamine',
}

amine_mols = {}
for _, row in df_smiles.iterrows():
    name = row['Compound_Name']
    smiles = row['SMILES']
    norm_name = name_map.get(name, name.lower().replace(' ', '_').replace('-', '_'))
    if pd.isna(smiles):
        continue
    smiles_clean = smiles.split('.')[0]
    mol = Chem.MolFromSmiles(smiles_clean)
    if mol is not None:
        amine_mols[norm_name] = mol

amines_needed = df_agg['Amine'].unique()
print(f"Amines needed: {len(amines_needed)}, parsed: {len(amine_mols)}")
print(f"Missing from SMILES: {set(amines_needed) - set(amine_mols.keys())}")

## 3. Compute Enzyme Representations (8 variants)

In [ ]:
THRESHOLDS = {
    'very_unique': 0.3,
    'unique': 0.5,
    'non_conserved': 0.8,
    'all_variable': 0.95,
}

for name, thresh in THRESHOLDS.items():
    n_pos = (df_core['conservation_score'] < thresh).sum()
    print(f"  {name} (< {thresh}): {n_pos} positions")


def get_nonconserved_embedding(enzyme_id, conservation_threshold, pooling='mean'):
    """Extract and pool per-residue embeddings at non-conserved positions."""
    if enzyme_id not in per_residue_embeddings or enzyme_id not in alignment_to_seq:
        return None
    embed = per_residue_embeddings[enzyme_id]
    aln_map = alignment_to_seq[enzyme_id]
    variable_aln_positions = df_core[
        df_core['conservation_score'] < conservation_threshold
    ]['alignment_position'].values
    seq_positions = []
    for aln_pos in variable_aln_positions:
        if aln_pos in aln_map:
            seq_pos = aln_map[aln_pos]
            if seq_pos < len(embed):
                seq_positions.append(seq_pos)
    if len(seq_positions) == 0:
        return None
    selected = embed[seq_positions]
    if pooling == 'mean':
        return selected.mean(axis=0)
    elif pooling == 'max':
        return selected.max(axis=0)
    elif pooling == 'mean_max':
        return np.concatenate([selected.mean(axis=0), selected.max(axis=0)])
    return selected.mean(axis=0)


def get_conserved_embedding(enzyme_id, conservation_threshold=0.95, pooling='mean'):
    """Extract per-residue embeddings at CONSERVED positions (>= threshold)."""
    if enzyme_id not in per_residue_embeddings or enzyme_id not in alignment_to_seq:
        return None
    embed = per_residue_embeddings[enzyme_id]
    aln_map = alignment_to_seq[enzyme_id]
    conserved_aln_positions = df_core[
        df_core['conservation_score'] >= conservation_threshold
    ]['alignment_position'].values
    seq_positions = []
    for aln_pos in conserved_aln_positions:
        if aln_pos in aln_map:
            seq_pos = aln_map[aln_pos]
            if seq_pos < len(embed):
                seq_positions.append(seq_pos)
    if len(seq_positions) == 0:
        return None
    selected = embed[seq_positions]
    if pooling == 'mean':
        return selected.mean(axis=0)
    return selected.mean(axis=0)

In [ ]:
# Precompute all 8 enzyme embedding dicts
enzyme_repr = {}

# 1. full_protein: mean of all residues (from Seqs_list_total.h5)
enzyme_repr['full_protein'] = {eid: emb for eid, emb in full_embeddings.items()}

# 2-5. Conservation-threshold variants (mean pooling)
for name, thresh in THRESHOLDS.items():
    d = {}
    for eid in overlap:
        emb = get_nonconserved_embedding(eid, thresh, pooling='mean')
        if emb is not None:
            d[eid] = emb
    enzyme_repr[name] = d

# 6. noncons_max: conservation < 0.5, max pooling
d = {}
for eid in overlap:
    emb = get_nonconserved_embedding(eid, 0.5, pooling='max')
    if emb is not None:
        d[eid] = emb
enzyme_repr['noncons_max'] = d

# 7. noncons_mean_max: conservation < 0.5, mean+max concatenated
d = {}
for eid in overlap:
    emb = get_nonconserved_embedding(eid, 0.5, pooling='mean_max')
    if emb is not None:
        d[eid] = emb
enzyme_repr['noncons_mean_max'] = d

# 8. cons_plus_noncons: conserved(mean) + non-conserved(mean) concatenated
d = {}
for eid in overlap:
    noncons = get_nonconserved_embedding(eid, 0.5, pooling='mean')
    cons = get_conserved_embedding(eid, 0.95, pooling='mean')
    if noncons is not None and cons is not None:
        d[eid] = np.concatenate([cons, noncons])
enzyme_repr['cons_plus_noncons'] = d

# Summary
print(f"{'Representation':<22} {'Enzymes':>8} {'Dims':>6}")
print("-" * 40)
for name, d in enzyme_repr.items():
    sample = list(d.values())[0]
    print(f"{name:<22} {len(d):>8} {sample.shape[0]:>6}")

## 4. Build Amine Representation (F_physchem_onehot)

In [ ]:
# Physicochemical descriptors
def compute_physicochemical(mol):
    return np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.NumAromaticRings(mol),
        Descriptors.NumAliphaticRings(mol),
        Descriptors.FractionCSP3(mol),
        Descriptors.HeavyAtomCount(mol),
        rdMolDescriptors.CalcNumAmideBonds(mol),
        Descriptors.NumValenceElectrons(mol),
        Descriptors.MaxPartialCharge(mol),
        Descriptors.MinPartialCharge(mol),
        Descriptors.BalabanJ(mol) if Descriptors.BalabanJ(mol) != 0 else 0.0,
    ], dtype=np.float32)

N_PHYSCHEM = 15

# Physicochemical features
repr_physchem = {}
for name, mol in amine_mols.items():
    repr_physchem[name] = compute_physicochemical(mol)
for a in amines_needed:
    if a not in repr_physchem:
        repr_physchem[a] = np.zeros(N_PHYSCHEM, dtype=np.float32)

# One-hot encoding
all_amines_sorted = sorted(amines_needed)
amine_to_idx = {a: i for i, a in enumerate(all_amines_sorted)}
n_amines = len(all_amines_sorted)

repr_onehot = {}
for name in amines_needed:
    vec = np.zeros(n_amines, dtype=np.float32)
    if name in amine_to_idx:
        vec[amine_to_idx[name]] = 1.0
    repr_onehot[name] = vec

# F_physchem_onehot = physicochemical + one-hot
amine_features = {}
for name in amines_needed:
    amine_features[name] = np.concatenate([repr_physchem[name], repr_onehot[name]])

sample = list(amine_features.values())[0]
print(f"F_physchem_onehot: {N_PHYSCHEM} physicochemical + {n_amines} one-hot = {len(sample)} dims")
print(f"Amines with features: {len(amine_features)}")

In [ ]:
# Build feature matrices for all 8 enzyme representations
feature_matrices = {}

for repr_name, enz_dict in enzyme_repr.items():
    X_list, y_list = [], []
    enzymes, amines = [], []
    for _, row in df_agg.iterrows():
        enzyme, amine = row['Enzyme'], row['Amine']
        if enzyme not in enz_dict or amine not in amine_features:
            continue
        features = np.concatenate([enz_dict[enzyme], amine_features[amine]])
        X_list.append(features)
        y_list.append(int(row['active']))
        enzymes.append(enzyme)
        amines.append(amine)
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    feature_matrices[repr_name] = (X, y, enzymes, amines)
    enz_dims = list(enz_dict.values())[0].shape[0]
    print(f"{repr_name}: X={X.shape}, enzyme_dims={enz_dims}, amine_dims={len(sample)}, active={y.mean():.1%}")

## 5. Split Analysis

Before training, examine the 10 enzyme hold-out splits:
- Which enzymes land in train/val/test for each split
- Activity rates per partition
- Per-enzyme activity profiles (how many amines are active)

In [ ]:
def enzyme_holdout_split_seed(X, y, enzymes, amines, seed, test_size=0.2, val_size=0.2):
    """Enzyme hold-out split with a specific random seed."""
    enzymes_arr = np.array(enzymes)
    amines_arr = np.array(amines)
    unique_enzymes = np.unique(enzymes_arr)
    profiles = np.array([y[enzymes_arr == e].mean() for e in unique_enzymes])
    bins = pd.cut(profiles, bins=5, labels=False)
    train_val_enz, test_enz = train_test_split(
        unique_enzymes, test_size=test_size, random_state=seed, stratify=bins
    )
    tv_profiles = np.array([y[enzymes_arr == e].mean() for e in train_val_enz])
    tv_bins = pd.cut(tv_profiles, bins=5, labels=False)
    train_enz, val_enz = train_test_split(
        train_val_enz, test_size=val_size, random_state=seed, stratify=tv_bins
    )
    train_mask = np.isin(enzymes_arr, train_enz)
    val_mask = np.isin(enzymes_arr, val_enz)
    test_mask = np.isin(enzymes_arr, test_enz)
    return {
        'X_train': X[train_mask], 'y_train': y[train_mask],
        'X_val': X[val_mask], 'y_val': y[val_mask],
        'X_test': X[test_mask], 'y_test': y[test_mask],
        'train_enz': train_enz, 'val_enz': val_enz, 'test_enz': test_enz,
        'test_enzymes_arr': enzymes_arr[test_mask],
        'test_amines_arr': amines_arr[test_mask],
    }

print("Split function ready.")

In [ ]:
# Use the 'unique' (conservation < 0.5) dataset as reference for split analysis
X_ref, y_ref, enz_ref, ami_ref = feature_matrices['unique']
enzymes_arr_ref = np.array(enz_ref)
unique_enzymes_ref = np.unique(enzymes_arr_ref)

# Compute per-enzyme activity profiles
enzyme_activity_profile = {}
for e in unique_enzymes_ref:
    mask = enzymes_arr_ref == e
    enzyme_activity_profile[e] = {
        'n_amines': mask.sum(),
        'n_active': y_ref[mask].sum(),
        'pct_active': y_ref[mask].mean(),
    }

print(f"Total enzymes: {len(unique_enzymes_ref)}")
print(f"Activity profiles:")
pcts = [v['pct_active'] for v in enzyme_activity_profile.values()]
print(f"  Mean activity rate: {np.mean(pcts):.1%}")
print(f"  Min: {np.min(pcts):.1%}, Max: {np.max(pcts):.1%}")
print(f"  Fully inactive enzymes: {sum(1 for p in pcts if p == 0)}")
print(f"  Highly active (>50%): {sum(1 for p in pcts if p > 0.5)}")

# Generate all 10 splits and record composition
split_info = []
enzyme_assignments = {}  # enzyme -> [partition_per_split]

for e in unique_enzymes_ref:
    enzyme_assignments[e] = []

all_splits = []
for i, seed in enumerate(SEEDS):
    split = enzyme_holdout_split_seed(X_ref, y_ref, enz_ref, ami_ref, seed=seed)
    all_splits.append(split)
    
    info = {
        'split_idx': i,
        'seed': seed,
        'n_train_enz': len(split['train_enz']),
        'n_val_enz': len(split['val_enz']),
        'n_test_enz': len(split['test_enz']),
        'n_train_samples': len(split['y_train']),
        'n_val_samples': len(split['y_val']),
        'n_test_samples': len(split['y_test']),
        'train_active_rate': split['y_train'].mean(),
        'val_active_rate': split['y_val'].mean(),
        'test_active_rate': split['y_test'].mean(),
    }
    split_info.append(info)
    
    for e in unique_enzymes_ref:
        if e in split['train_enz']:
            enzyme_assignments[e].append('train')
        elif e in split['val_enz']:
            enzyme_assignments[e].append('val')
        elif e in split['test_enz']:
            enzyme_assignments[e].append('test')
        else:
            enzyme_assignments[e].append('missing')

df_split_info = pd.DataFrame(split_info)
print("\nSplit composition:")
print(df_split_info[['split_idx', 'n_train_enz', 'n_val_enz', 'n_test_enz',
                      'train_active_rate', 'val_active_rate', 'test_active_rate']].to_string(index=False, float_format='%.3f'))

In [ ]:
# Heatmap: enzyme -> split assignment across 10 splits
# Sort enzymes by activity profile
sorted_enzymes = sorted(unique_enzymes_ref, key=lambda e: enzyme_activity_profile[e]['pct_active'])

# Build assignment matrix
partition_to_num = {'train': 0, 'val': 1, 'test': 2, 'missing': -1}
assignment_matrix = np.zeros((len(sorted_enzymes), N_SPLITS))
for i, e in enumerate(sorted_enzymes):
    for j, part in enumerate(enzyme_assignments[e]):
        assignment_matrix[i, j] = partition_to_num[part]

fig, axes = plt.subplots(1, 2, figsize=(18, 10), gridspec_kw={'width_ratios': [3, 1]})

# Left: assignment heatmap
ax = axes[0]
cmap = plt.cm.colors.ListedColormap(['#3498db', '#f39c12', '#e74c3c'])
im = ax.imshow(assignment_matrix, aspect='auto', cmap=cmap, vmin=0, vmax=2)
ax.set_xlabel('Split Index', fontsize=12)
ax.set_ylabel('Enzyme (sorted by activity rate)', fontsize=12)
ax.set_title('Enzyme -> Partition Assignment', fontsize=14, fontweight='bold')
ax.set_xticks(range(N_SPLITS))
ax.set_xticklabels([str(i) for i in range(N_SPLITS)])
# Sparse y-axis labels
tick_step = max(1, len(sorted_enzymes) // 20)
ax.set_yticks(range(0, len(sorted_enzymes), tick_step))
ax.set_yticklabels([sorted_enzymes[i] for i in range(0, len(sorted_enzymes), tick_step)], fontsize=7)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#3498db', label='Train'),
                   Patch(facecolor='#f39c12', label='Val'),
                   Patch(facecolor='#e74c3c', label='Test')]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

# Right: enzyme activity profile
ax2 = axes[1]
pcts_sorted = [enzyme_activity_profile[e]['pct_active'] for e in sorted_enzymes]
ax2.barh(range(len(sorted_enzymes)), pcts_sorted, color='#2ecc71', alpha=0.7)
ax2.set_xlabel('Activity Rate', fontsize=12)
ax2.set_title('Per-Enzyme Activity', fontsize=14, fontweight='bold')
ax2.set_ylim(-0.5, len(sorted_enzymes) - 0.5)
ax2.set_yticks([])
ax2.axvline(0.5, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(COMBO_DIR / 'split_enzyme_assignment.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMBO_DIR / 'split_enzyme_assignment.png'}")

In [ ]:
# Split analysis: activity rates and sample counts per split
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Activity rates per partition
ax = axes[0]
x = np.arange(N_SPLITS)
width = 0.25
ax.bar(x - width, df_split_info['train_active_rate'], width, label='Train', color='#3498db')
ax.bar(x, df_split_info['val_active_rate'], width, label='Val', color='#f39c12')
ax.bar(x + width, df_split_info['test_active_rate'], width, label='Test', color='#e74c3c')
ax.set_xlabel('Split Index', fontsize=11)
ax.set_ylabel('Active Fraction', fontsize=11)
ax.set_title('Activity Rate per Partition', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Sample counts per partition
ax = axes[1]
ax.bar(x - width, df_split_info['n_train_samples'], width, label='Train', color='#3498db')
ax.bar(x, df_split_info['n_val_samples'], width, label='Val', color='#f39c12')
ax.bar(x + width, df_split_info['n_test_samples'], width, label='Test', color='#e74c3c')
ax.set_xlabel('Split Index', fontsize=11)
ax.set_ylabel('Number of Samples', fontsize=11)
ax.set_title('Samples per Partition', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Number of enzymes per partition
ax = axes[2]
ax.bar(x - width, df_split_info['n_train_enz'], width, label='Train', color='#3498db')
ax.bar(x, df_split_info['n_val_enz'], width, label='Val', color='#f39c12')
ax.bar(x + width, df_split_info['n_test_enz'], width, label='Test', color='#e74c3c')
ax.set_xlabel('Split Index', fontsize=11)
ax.set_ylabel('Number of Enzymes', fontsize=11)
ax.set_title('Enzymes per Partition', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(COMBO_DIR / 'split_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMBO_DIR / 'split_analysis.png'}")

# Save split composition
df_split_info.to_csv(COMBO_DIR / 'split_analysis.csv', index=False)
print(f"Saved: {COMBO_DIR / 'split_analysis.csv'}")

## 6. Train with Log Loss Tracking (8 enzyme repr x 10 splits = 80 experiments)

In [ ]:
def train_xgb_with_logloss(split_data):
    """
    Train regularized XGBoost, tracking train+val log loss per round.
    Returns metrics dict and log loss curves.
    """
    X_train, y_train = split_data['X_train'], split_data['y_train']
    X_val, y_val = split_data['X_val'], split_data['y_val']
    X_test, y_test = split_data['X_test'], split_data['y_test']
    
    n_neg = (y_train == 0).sum()
    n_pos = max((y_train == 1).sum(), 1)
    
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        scale_pos_weight=n_neg / n_pos,
        reg_alpha=1.0, reg_lambda=5.0,
        subsample=0.7, colsample_bytree=0.7,
        min_child_weight=5,
        random_state=42, early_stopping_rounds=30,
        eval_metric='logloss', n_jobs=-1
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False
    )
    
    evals = model.evals_result()
    train_logloss = evals['validation_0']['logloss']
    val_logloss = evals['validation_1']['logloss']
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_proba_train = model.predict_proba(X_train)[:, 1]
    y_proba_val = model.predict_proba(X_val)[:, 1]
    
    metrics = {
        'train_logloss': log_loss(y_train, y_proba_train),
        'val_logloss': log_loss(y_val, y_proba_val),
        'test_logloss': log_loss(y_test, y_proba),
        'logloss_gap': log_loss(y_val, y_proba_val) - log_loss(y_train, y_proba_train),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'accuracy': accuracy_score(y_test, y_pred),
        'train_acc': model.score(X_train, y_train),
        'val_acc': model.score(X_val, y_val),
        'n_rounds': len(train_logloss),
    }
    
    return metrics, train_logloss, val_logloss, model

print("Training function ready.")

In [ ]:
%%time

# Train all 8 enzyme representations x 10 splits = 80 experiments
all_results = []
all_logloss_curves = {}  # {repr_name: [(train_curve, val_curve), ...] per split}
all_trained_splits = {}  # {repr_name: [split_data, ...]} for later diagnostics
all_models = {}          # {repr_name: [model, ...]}

repr_names = list(feature_matrices.keys())

for repr_name in repr_names:
    X, y, enzymes, amines = feature_matrices[repr_name]
    all_logloss_curves[repr_name] = []
    all_trained_splits[repr_name] = []
    all_models[repr_name] = []
    
    for i, seed in enumerate(SEEDS):
        split = enzyme_holdout_split_seed(X, y, enzymes, amines, seed=seed)
        metrics, train_ll, val_ll, model = train_xgb_with_logloss(split)
        metrics['representation'] = repr_name
        metrics['split_idx'] = i
        metrics['split_seed'] = seed
        all_results.append(metrics)
        all_logloss_curves[repr_name].append((train_ll, val_ll))
        all_trained_splits[repr_name].append(split)
        all_models[repr_name].append(model)
    
    code_results = [r for r in all_results if r['representation'] == repr_name]
    roc_vals = [r['roc_auc'] for r in code_results]
    gap_vals = [r['logloss_gap'] for r in code_results]
    print(f"{repr_name}: ROC-AUC={np.mean(roc_vals):.3f}+/-{np.std(roc_vals):.3f}, "
          f"LogLoss gap={np.mean(gap_vals):.3f}+/-{np.std(gap_vals):.3f}")

df_results = pd.DataFrame(all_results)
print(f"\nTotal experiments: {len(df_results)} ({len(repr_names)} repr x {N_SPLITS} splits)")
print("Done!")

## 7. Performance Comparison

In [ ]:
# Summary statistics per representation
summary_rows = []
for repr_name in repr_names:
    mask = df_results['representation'] == repr_name
    df_sub = df_results[mask]
    X_shape = feature_matrices[repr_name][0].shape
    enz_dims = list(enzyme_repr[repr_name].values())[0].shape[0]
    
    row = {
        'representation': repr_name,
        'total_dims': X_shape[1],
        'enzyme_dims': enz_dims,
        'roc_auc_mean': df_sub['roc_auc'].mean(),
        'roc_auc_std': df_sub['roc_auc'].std(),
        'pr_auc_mean': df_sub['pr_auc'].mean(),
        'pr_auc_std': df_sub['pr_auc'].std(),
        'f1_mean': df_sub['f1'].mean(),
        'f1_std': df_sub['f1'].std(),
        'logloss_gap_mean': df_sub['logloss_gap'].mean(),
        'logloss_gap_std': df_sub['logloss_gap'].std(),
        'val_logloss_mean': df_sub['val_logloss'].mean(),
        'train_logloss_mean': df_sub['train_logloss'].mean(),
        'train_acc_mean': df_sub['train_acc'].mean(),
        'val_acc_mean': df_sub['val_acc'].mean(),
        'accuracy_mean': df_sub['accuracy'].mean(),
    }
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)

print("Summary (mean +/- std over 10 splits):")
print("=" * 110)
print(f"{'Repr':<22} {'Dims':>5} {'ROC-AUC':>14} {'PR-AUC':>14} {'F1':>12} {'LL Gap':>12} {'Val LL':>8}")
print("-" * 110)
for _, r in df_summary.sort_values('pr_auc_mean', ascending=False).iterrows():
    print(f"{r['representation']:<22} {r['total_dims']:>5.0f} "
          f"{r['roc_auc_mean']:.3f}+/-{r['roc_auc_std']:.3f} "
          f"{r['pr_auc_mean']:.3f}+/-{r['pr_auc_std']:.3f} "
          f"{r['f1_mean']:.3f}+/-{r['f1_std']:.3f} "
          f"{r['logloss_gap_mean']:>+.3f}+/-{r['logloss_gap_std']:.3f} "
          f"{r['val_logloss_mean']:>.3f}")

In [ ]:
# Bar charts: ROC-AUC, PR-AUC, F1, log loss gap
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Sort by PR-AUC for consistent ordering
order = df_summary.sort_values('pr_auc_mean', ascending=False)['representation'].values
x = np.arange(len(order))

metrics_plot = [
    ('roc_auc', 'ROC-AUC', '#3498db'),
    ('pr_auc', 'PR-AUC', '#2ecc71'),
    ('f1', 'F1 Score', '#e67e22'),
    ('logloss_gap', 'Log Loss Gap (val - train)', '#e74c3c'),
]

for ax, (metric, title, color) in zip(axes.flat, metrics_plot):
    means = [df_summary[df_summary['representation'] == r][f'{metric}_mean'].values[0] for r in order]
    stds = [df_summary[df_summary['representation'] == r][f'{metric}_std'].values[0] for r in order]
    
    bars = ax.bar(x, means, yerr=stds, color=color, alpha=0.7,
                  edgecolor='black', linewidth=0.5, capsize=4)
    
    for i, (m, s) in enumerate(zip(means, stds)):
        ax.text(i, m + s + 0.01, f'{m:.3f}', ha='center', fontsize=8, fontweight='bold')
    
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    if metric == 'logloss_gap':
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    elif metric == 'roc_auc':
        ax.axhline(0.5, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)

plt.suptitle('Enzyme Representation Comparison (F_physchem_onehot amine, Regularized XGBoost, 10 splits)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(COMBO_DIR / 'enzyme_repr_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMBO_DIR / 'enzyme_repr_comparison_bars.png'}")

In [ ]:
# Box plots: variance across splits
fig, axes = plt.subplots(1, 4, figsize=(22, 6))

for ax, metric, title in zip(axes,
    ['roc_auc', 'pr_auc', 'f1', 'logloss_gap'],
    ['ROC-AUC', 'PR-AUC', 'F1', 'Log Loss Gap']):
    
    data_to_plot = []
    labels = []
    for r in order:
        vals = df_results[df_results['representation'] == r][metric].values
        data_to_plot.append(vals)
        labels.append(r)
    
    bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
    colors_box = plt.cm.Set2(np.linspace(0, 1, len(order)))
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Variance Across 10 Enzyme Hold-Out Splits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(COMBO_DIR / 'enzyme_repr_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMBO_DIR / 'enzyme_repr_boxplots.png'}")

In [ ]:
# Log loss convergence: 8-panel plot
fig, axes = plt.subplots(2, 4, figsize=(24, 10))

for idx, repr_name in enumerate(repr_names):
    ax = axes.flat[idx]
    curves = all_logloss_curves[repr_name]
    
    for i, (train_ll, val_ll) in enumerate(curves):
        rounds = np.arange(len(train_ll))
        ax.plot(rounds, train_ll, color='#3498db', alpha=0.3, linewidth=0.8)
        ax.plot(rounds, val_ll, color='#e74c3c', alpha=0.3, linewidth=0.8)
    
    # Mean curves
    min_len = min(len(c[0]) for c in curves)
    train_mean = np.mean([c[0][:min_len] for c in curves], axis=0)
    val_mean = np.mean([c[1][:min_len] for c in curves], axis=0)
    rounds = np.arange(min_len)
    ax.plot(rounds, train_mean, color='#2980b9', linewidth=2.5, label='Train (mean)')
    ax.plot(rounds, val_mean, color='#c0392b', linewidth=2.5, label='Val (mean)')
    
    final_gap = val_mean[-1] - train_mean[-1]
    ax.set_title(f"{repr_name}\nFinal gap: {final_gap:.3f}", fontsize=10, fontweight='bold')
    ax.set_xlabel('Boosting Round', fontsize=8)
    ax.set_ylabel('Log Loss', fontsize=8)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Log Loss Convergence: Train vs Validation (10 splits overlaid)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(COMBO_DIR / 'logloss_convergence_enzyme.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMBO_DIR / 'logloss_convergence_enzyme.png'}")

## 8. Split-Performance Diagnostics

Correlate split properties with model performance:
- Does test set activity rate predict ROC-AUC?
- Which splits are hardest/easiest across all representations?
- Per-enzyme prediction difficulty

In [ ]:
# Scatter: test activity rate vs ROC-AUC across splits, colored by representation
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

cmap_repr = plt.cm.tab10(np.linspace(0, 1, len(repr_names)))
repr_colors = {r: cmap_repr[i] for i, r in enumerate(repr_names)}

for ax, (metric, title) in zip(axes, 
    [('roc_auc', 'ROC-AUC'), ('pr_auc', 'PR-AUC'), ('f1', 'F1')]):
    
    for repr_name in repr_names:
        mask = df_results['representation'] == repr_name
        df_sub = df_results[mask].copy()
        # Merge test activity rates from split_info
        df_sub = df_sub.merge(df_split_info[['split_idx', 'test_active_rate']], on='split_idx')
        
        ax.scatter(df_sub['test_active_rate'], df_sub[metric],
                   color=repr_colors[repr_name], label=repr_name,
                   alpha=0.7, s=40, edgecolors='black', linewidth=0.3)
    
    ax.set_xlabel('Test Set Activity Rate', fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f'{title} vs Test Activity Rate', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

# Single legend outside
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center right', fontsize=9, bbox_to_anchor=(1.12, 0.5))

plt.tight_layout()
plt.savefig(COMBO_DIR / 'split_vs_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMBO_DIR / 'split_vs_performance.png'}")

In [ ]:
# Which splits are hardest/easiest? Average ROC-AUC across all representations per split
split_perf = df_results.groupby('split_idx').agg(
    roc_auc_mean=('roc_auc', 'mean'),
    roc_auc_std=('roc_auc', 'std'),
    pr_auc_mean=('pr_auc', 'mean'),
    f1_mean=('f1', 'mean'),
).reset_index()
split_perf = split_perf.merge(df_split_info[['split_idx', 'seed', 'test_active_rate', 'n_test_enz']], on='split_idx')
split_perf = split_perf.sort_values('roc_auc_mean')

print("Split difficulty ranking (averaged across all 8 enzyme representations):")
print("=" * 90)
print(f"{'Split':>5} {'Seed':>6} {'ROC-AUC':>14} {'PR-AUC':>9} {'F1':>7} {'Test Active':>12} {'Test Enz':>9}")
print("-" * 90)
for _, r in split_perf.iterrows():
    diff_label = 'HARD' if r['roc_auc_mean'] < split_perf['roc_auc_mean'].median() else 'EASY'
    print(f"{r['split_idx']:>5.0f} {r['seed']:>6.0f} "
          f"{r['roc_auc_mean']:.3f}+/-{r['roc_auc_std']:.3f} "
          f"{r['pr_auc_mean']:.3f}     {r['f1_mean']:.3f} "
          f"{r['test_active_rate']:>11.3f} {r['n_test_enz']:>9.0f}   {diff_label}")

In [ ]:
# Per-enzyme prediction difficulty: for each test enzyme, compute accuracy across splits and representations
enzyme_difficulty = []

for repr_name in repr_names:
    for i in range(N_SPLITS):
        split = all_trained_splits[repr_name][i]
        model = all_models[repr_name][i]
        
        test_enz_arr = split['test_enzymes_arr']
        y_test = split['y_test']
        y_proba = model.predict_proba(split['X_test'])[:, 1]
        y_pred = model.predict(split['X_test'])
        
        for enz in np.unique(test_enz_arr):
            enz_mask = test_enz_arr == enz
            if enz_mask.sum() == 0:
                continue
            enzyme_difficulty.append({
                'enzyme': enz,
                'representation': repr_name,
                'split_idx': i,
                'n_pairs': enz_mask.sum(),
                'accuracy': accuracy_score(y_test[enz_mask], y_pred[enz_mask]),
                'roc_auc': roc_auc_score(y_test[enz_mask], y_proba[enz_mask]) if len(np.unique(y_test[enz_mask])) > 1 else np.nan,
                'true_active_rate': y_test[enz_mask].mean(),
            })

df_enz_diff = pd.DataFrame(enzyme_difficulty)
print(f"Enzyme-level predictions: {len(df_enz_diff)} entries")

# Average across representations and splits for each enzyme
enz_avg = df_enz_diff.groupby('enzyme').agg(
    mean_accuracy=('accuracy', 'mean'),
    std_accuracy=('accuracy', 'std'),
    mean_roc_auc=('roc_auc', 'mean'),
    n_appearances=('split_idx', 'count'),
    true_active_rate=('true_active_rate', 'mean'),
).reset_index().sort_values('mean_accuracy')

print("\nHardest enzymes to predict (lowest accuracy):")
print(enz_avg.head(15).to_string(index=False, float_format='%.3f'))

print("\nEasiest enzymes to predict (highest accuracy):")
print(enz_avg.tail(10).to_string(index=False, float_format='%.3f'))

In [ ]:
# Enzyme difficulty plot
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: bar chart of per-enzyme accuracy (sorted)
ax = axes[0]
enz_sorted = enz_avg.sort_values('mean_accuracy')
colors_acc = ['#e74c3c' if a < 0.6 else '#f39c12' if a < 0.75 else '#2ecc71' 
              for a in enz_sorted['mean_accuracy']]
ax.barh(range(len(enz_sorted)), enz_sorted['mean_accuracy'], 
        xerr=enz_sorted['std_accuracy'], color=colors_acc, alpha=0.8, capsize=2)
ax.set_yticks(range(len(enz_sorted)))
ax.set_yticklabels(enz_sorted['enzyme'], fontsize=6)
ax.set_xlabel('Mean Accuracy', fontsize=11)
ax.set_title('Per-Enzyme Prediction Accuracy\n(across all splits & representations)', fontsize=12, fontweight='bold')
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3, axis='x')

# Right: scatter of enzyme activity rate vs prediction accuracy
ax = axes[1]
scatter = ax.scatter(enz_avg['true_active_rate'], enz_avg['mean_accuracy'],
                     c=enz_avg['std_accuracy'], cmap='RdYlGn_r', 
                     s=60, alpha=0.8, edgecolors='black', linewidth=0.5)
plt.colorbar(scatter, ax=ax, label='Std Accuracy (variance)')

# Label hardest enzymes
for _, row in enz_avg.head(5).iterrows():
    ax.annotate(row['enzyme'], (row['true_active_rate'], row['mean_accuracy']),
                fontsize=7, ha='left', va='bottom', alpha=0.8)

ax.set_xlabel('True Activity Rate', fontsize=11)
ax.set_ylabel('Mean Prediction Accuracy', fontsize=11)
ax.set_title('Enzyme Activity Rate vs Prediction Accuracy', fontsize=12, fontweight='bold')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(COMBO_DIR / 'enzyme_difficulty.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMBO_DIR / 'enzyme_difficulty.png'}")

In [ ]:
# Heatmap: representation x split performance (using matplotlib imshow)
pivot_roc = df_results.pivot_table(index='representation', columns='split_idx', values='roc_auc')
pivot_roc = pivot_roc.loc[order]  # Use PR-AUC sorted order

pivot_pr = df_results.pivot_table(index='representation', columns='split_idx', values='pr_auc')
pivot_pr = pivot_pr.loc[order]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, pivot, title, label in [
    (axes[0], pivot_roc, 'ROC-AUC: Representation x Split', 'ROC-AUC'),
    (axes[1], pivot_pr, 'PR-AUC: Representation x Split', 'PR-AUC')
]:
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn',
                   vmin=pivot.values.min() - 0.02, vmax=pivot.values.max() + 0.02)
    
    # Add text annotations
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=7,
                    color='black' if 0.4 < (val - pivot.values.min()) / (pivot.values.max() - pivot.values.min()) < 0.7 else 'white')
    
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels([str(c) for c in pivot.columns])
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_xlabel('Split Index', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax, label=label, shrink=0.8)

axes[0].set_ylabel('Enzyme Representation', fontsize=11)

plt.tight_layout()
plt.savefig(COMBO_DIR / 'repr_split_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMBO_DIR / 'repr_split_heatmap.png'}")

## 8b. Feature Importance Analysis

For each enzyme representation, examine what the XGBoost model is actually using:
1. **Enzyme vs amine block importance** — how much weight goes to each feature block?
2. **Amine feature breakdown** — which of the 15 physicochemical + 26 one-hot features matter?
3. **Stability across splits** — are the same features important regardless of split?

In [ ]:
# Feature importance: enzyme block vs amine block across all 8 representations x 10 splits
PHYSCHEM_NAMES = [
    'MolWt', 'LogP', 'TPSA', 'HBD', 'HBA', 'RotBonds', 'AromaticRings',
    'AliphaticRings', 'FractionCSP3', 'HeavyAtoms', 'AmideBonds',
    'ValenceElectrons', 'MaxPartialCharge', 'MinPartialCharge', 'BalabanJ'
]
AMINE_FEATURE_NAMES = PHYSCHEM_NAMES + [f'onehot_{a}' for a in all_amines_sorted]

importance_records = []

for repr_name in repr_names:
    enz_dims = list(enzyme_repr[repr_name].values())[0].shape[0]
    amine_dims = len(sample)  # 41
    
    for i in range(N_SPLITS):
        model = all_models[repr_name][i]
        imp = model.feature_importances_
        
        # Block-level importance
        enzyme_imp = imp[:enz_dims].sum()
        amine_imp = imp[enz_dims:].sum()
        total_imp = imp.sum()
        
        # Per-amine-feature importance
        amine_feature_imp = imp[enz_dims:]  # 41 values
        
        importance_records.append({
            'representation': repr_name,
            'split_idx': i,
            'enzyme_dims': enz_dims,
            'enzyme_frac': enzyme_imp / total_imp,
            'amine_frac': amine_imp / total_imp,
            'amine_feature_imp': amine_feature_imp,
            'top_enzyme_features': np.argsort(imp[:enz_dims])[-10:][::-1],  # top 10 enzyme dims
        })

df_imp = pd.DataFrame(importance_records)

# Summary: enzyme vs amine fraction per representation
print("Feature Importance: Enzyme vs Amine Block")
print("=" * 75)
print(f"{'Representation':<22} {'Enz dims':>8} {'Enzyme %':>12} {'Amine %':>12}")
print("-" * 75)
for repr_name in repr_names:
    mask = df_imp['representation'] == repr_name
    enz_frac = df_imp[mask]['enzyme_frac'].mean()
    ami_frac = df_imp[mask]['amine_frac'].mean()
    enz_dims = df_imp[mask]['enzyme_dims'].iloc[0]
    print(f"{repr_name:<22} {enz_dims:>8} {enz_frac:>11.1%} {ami_frac:>11.1%}")

In [ ]:
# Stacked bar: enzyme vs amine importance fraction per representation
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(repr_names))
width = 0.6

enzyme_fracs = [df_imp[df_imp['representation'] == r]['enzyme_frac'].mean() for r in repr_names]
amine_fracs = [df_imp[df_imp['representation'] == r]['amine_frac'].mean() for r in repr_names]
enzyme_stds = [df_imp[df_imp['representation'] == r]['enzyme_frac'].std() for r in repr_names]
amine_stds = [df_imp[df_imp['representation'] == r]['amine_frac'].std() for r in repr_names]

ax.bar(x, enzyme_fracs, width, label='Enzyme features', color='#3498db', alpha=0.8)
ax.bar(x, amine_fracs, width, bottom=enzyme_fracs, label='Amine features', color='#e74c3c', alpha=0.8)

# Annotate amine fraction
for i, (ef, af) in enumerate(zip(enzyme_fracs, amine_fracs)):
    ax.text(i, ef + af/2, f'{af:.1%}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    ax.text(i, ef/2, f'{ef:.1%}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')

ax.set_xticks(x)
ax.set_xticklabels(repr_names, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Fraction of Total Feature Importance', fontsize=12)
ax.set_title('Feature Importance: Enzyme vs Amine Block (mean across 10 splits)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(COMBO_DIR / 'feature_importance_blocks.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMBO_DIR / 'feature_importance_blocks.png'}")

In [ ]:
# Amine feature breakdown: which of the 41 amine features matter most?
# Average amine feature importances across all splits for each representation

fig, axes = plt.subplots(2, 4, figsize=(26, 12))

for idx, repr_name in enumerate(repr_names):
    ax = axes.flat[idx]
    mask = df_imp['representation'] == repr_name
    
    # Stack amine feature importances across splits
    amine_imps = np.array([r for r in df_imp[mask]['amine_feature_imp']])  # (10, 41)
    mean_imp = amine_imps.mean(axis=0)
    std_imp = amine_imps.std(axis=0)
    
    # Normalize to fraction of amine importance
    total_amine = mean_imp.sum()
    if total_amine > 0:
        mean_frac = mean_imp / total_amine
        std_frac = std_imp / total_amine
    else:
        mean_frac = mean_imp
        std_frac = std_imp
    
    # Color: physicochemical in blue, one-hot in orange
    colors = ['#3498db'] * N_PHYSCHEM + ['#e67e22'] * n_amines
    
    bars = ax.barh(range(len(AMINE_FEATURE_NAMES)), mean_frac, 
                   xerr=std_frac, color=colors, alpha=0.8, capsize=2)
    
    ax.set_yticks(range(len(AMINE_FEATURE_NAMES)))
    ax.set_yticklabels(AMINE_FEATURE_NAMES, fontsize=6)
    ax.set_xlabel('Fraction of Amine Importance', fontsize=9)
    ax.set_title(f'{repr_name}', fontsize=10, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Amine Feature Importance Breakdown (blue=physicochemical, orange=one-hot)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(COMBO_DIR / 'amine_feature_importance_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMBO_DIR / 'amine_feature_importance_breakdown.png'}")

In [ ]:
# Aggregated view: average amine feature importance across ALL representations and splits
all_amine_imps = np.array([r for r in df_imp['amine_feature_imp']])  # (80, 41)
global_mean = all_amine_imps.mean(axis=0)
global_std = all_amine_imps.std(axis=0)

# Normalize
global_frac = global_mean / global_mean.sum()
global_frac_std = global_std / global_mean.sum()

# Sort by importance
sort_idx = np.argsort(global_frac)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: ranked amine features
ax = axes[0]
sorted_names = [AMINE_FEATURE_NAMES[i] for i in sort_idx]
sorted_fracs = global_frac[sort_idx]
sorted_stds = global_frac_std[sort_idx]
sorted_colors = ['#3498db' if i < N_PHYSCHEM else '#e67e22' for i in sort_idx]

ax.barh(range(len(sorted_names)), sorted_fracs, xerr=sorted_stds,
        color=sorted_colors, alpha=0.8, capsize=2)
ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels(sorted_names, fontsize=8)
ax.set_xlabel('Fraction of Amine Feature Importance', fontsize=11)
ax.set_title('Amine Features Ranked by Importance\n(averaged across all repr & splits)', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#3498db', label='Physicochemical'),
                   Patch(color='#e67e22', label='One-hot (amine identity)')],
          fontsize=10, loc='lower right')

# Right: physicochemical vs one-hot total importance
ax = axes[1]
physchem_total = global_frac[:N_PHYSCHEM].sum()
onehot_total = global_frac[N_PHYSCHEM:].sum()

# Per representation
physchem_per_repr = []
onehot_per_repr = []
for repr_name in repr_names:
    mask = df_imp['representation'] == repr_name
    amine_imps_r = np.array([r for r in df_imp[mask]['amine_feature_imp']])
    mean_r = amine_imps_r.mean(axis=0)
    total_r = mean_r.sum()
    if total_r > 0:
        physchem_per_repr.append(mean_r[:N_PHYSCHEM].sum() / total_r)
        onehot_per_repr.append(mean_r[N_PHYSCHEM:].sum() / total_r)
    else:
        physchem_per_repr.append(0)
        onehot_per_repr.append(0)

x_r = np.arange(len(repr_names))
ax.bar(x_r, physchem_per_repr, 0.6, label='Physicochemical (15)', color='#3498db', alpha=0.8)
ax.bar(x_r, onehot_per_repr, 0.6, bottom=physchem_per_repr, label='One-hot (26)', color='#e67e22', alpha=0.8)

for i, (p, o) in enumerate(zip(physchem_per_repr, onehot_per_repr)):
    ax.text(i, p/2, f'{p:.0%}', ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    ax.text(i, p + o/2, f'{o:.0%}', ha='center', va='center', fontsize=8, fontweight='bold', color='white')

ax.set_xticks(x_r)
ax.set_xticklabels(repr_names, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Fraction of Amine Block Importance', fontsize=11)
ax.set_title('Physicochemical vs One-Hot Within Amine Block', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(COMBO_DIR / 'amine_feature_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMBO_DIR / 'amine_feature_summary.png'}")

In [ ]:
# Top enzyme embedding dimensions: which ProtT5 dimensions are most used?
# Focus on top 3 representations
top_reprs = ['cons_plus_noncons', 'noncons_max', 'unique']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, repr_name in zip(axes, top_reprs):
    enz_dims = list(enzyme_repr[repr_name].values())[0].shape[0]
    
    # Collect enzyme feature importances across splits
    enz_imps = []
    for i in range(N_SPLITS):
        model = all_models[repr_name][i]
        imp = model.feature_importances_[:enz_dims]
        enz_imps.append(imp)
    
    enz_imps = np.array(enz_imps)  # (10, enz_dims)
    mean_imp = enz_imps.mean(axis=0)
    
    # How many dimensions carry meaningful importance?
    sorted_imp = np.sort(mean_imp)[::-1]
    cumulative = np.cumsum(sorted_imp) / sorted_imp.sum()
    
    ax.plot(range(len(cumulative)), cumulative, color='#3498db', linewidth=2)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='50%')
    ax.axhline(0.9, color='gray', linestyle=':', alpha=0.5, label='90%')
    
    # Find how many dims for 50% and 90%
    n_50 = np.searchsorted(cumulative, 0.5) + 1
    n_90 = np.searchsorted(cumulative, 0.9) + 1
    ax.axvline(n_50, color='#e74c3c', linestyle='--', alpha=0.7)
    ax.axvline(n_90, color='#e67e22', linestyle='--', alpha=0.7)
    
    ax.set_xlabel('Number of Enzyme Dimensions (ranked)', fontsize=11)
    ax.set_ylabel('Cumulative Importance', fontsize=11)
    ax.set_title(f'{repr_name} ({enz_dims} dims)\\n50% at {n_50} dims, 90% at {n_90} dims',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, min(500, enz_dims))

plt.suptitle('Enzyme Embedding: Cumulative Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(COMBO_DIR / 'enzyme_cumulative_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMBO_DIR / 'enzyme_cumulative_importance.png'}")

# Print: fraction of enzyme dims that are essentially unused
print("\nEnzyme dimension utilization:")
print(f"{'Representation':<22} {'Total dims':>10} {'Dims for 50%':>13} {'Dims for 90%':>13} {'% used (>0.001)':>16}")
print("-" * 80)
for repr_name in repr_names:
    enz_dims = list(enzyme_repr[repr_name].values())[0].shape[0]
    enz_imps = []
    for i in range(N_SPLITS):
        model = all_models[repr_name][i]
        enz_imps.append(model.feature_importances_[:enz_dims])
    mean_imp = np.array(enz_imps).mean(axis=0)
    sorted_imp = np.sort(mean_imp)[::-1]
    cumulative = np.cumsum(sorted_imp) / sorted_imp.sum()
    n_50 = np.searchsorted(cumulative, 0.5) + 1
    n_90 = np.searchsorted(cumulative, 0.9) + 1
    n_used = (mean_imp > 0.001).sum()
    print(f"{repr_name:<22} {enz_dims:>10} {n_50:>13} {n_90:>13} {n_used:>10} ({100*n_used/enz_dims:.0f}%)")

## 9. Save Results

In [ ]:
# Save full results (80 rows: 8 repr x 10 splits)
df_results.to_csv(COMBO_DIR / 'enzyme_amine_combination_results.csv', index=False)
print(f"Saved: {COMBO_DIR / 'enzyme_amine_combination_results.csv'} ({len(df_results)} rows)")

# Save summary table (8 rows)
df_summary.to_csv(COMBO_DIR / 'enzyme_amine_combination_summary.csv', index=False)
print(f"Saved: {COMBO_DIR / 'enzyme_amine_combination_summary.csv'} ({len(df_summary)} rows)")

# Save enzyme difficulty data
enz_avg.to_csv(COMBO_DIR / 'enzyme_difficulty.csv', index=False)
print(f"Saved: {COMBO_DIR / 'enzyme_difficulty.csv'}")

print(f"\nAll outputs saved to: {COMBO_DIR}")

In [ ]:
# Final summary
print("=" * 90)
print("ENZYME x AMINE COMBINATION ANALYSIS - FINAL SUMMARY")
print("=" * 90)
print(f"\nModel: Regularized XGBoost (max_depth=3, reg_alpha=1.0, reg_lambda=5.0, subsample=0.7)")
print(f"Amine: F_physchem_onehot (physicochemical + one-hot, {len(sample)} dims)")
print(f"Splits: {N_SPLITS} random enzyme hold-out splits")
print()

# Ranked table
df_ranked = df_summary.sort_values('pr_auc_mean', ascending=False)
print(f"{'Rank':>4} {'Representation':<22} {'Dims':>5} {'ROC-AUC':>14} {'PR-AUC':>14} {'F1':>12} {'LL Gap':>12}")
print("-" * 90)
for rank, (_, r) in enumerate(df_ranked.iterrows(), 1):
    marker = ' ***' if rank == 1 else ''
    print(f"{rank:>4} {r['representation']:<22} {r['total_dims']:>5.0f} "
          f"{r['roc_auc_mean']:.3f}+/-{r['roc_auc_std']:.3f} "
          f"{r['pr_auc_mean']:.3f}+/-{r['pr_auc_std']:.3f} "
          f"{r['f1_mean']:.3f}+/-{r['f1_std']:.3f} "
          f"{r['logloss_gap_mean']:>+.3f}+/-{r['logloss_gap_std']:.3f}{marker}")

best = df_ranked.iloc[0]
print(f"\nBEST ENZYME REPRESENTATION: {best['representation']}")
print(f"  Total dims:  {best['total_dims']:.0f} (enzyme: {best['enzyme_dims']:.0f} + amine: {len(sample)})")
print(f"  ROC-AUC:     {best['roc_auc_mean']:.3f} +/- {best['roc_auc_std']:.3f}")
print(f"  PR-AUC:      {best['pr_auc_mean']:.3f} +/- {best['pr_auc_std']:.3f}")
print(f"  F1:          {best['f1_mean']:.3f} +/- {best['f1_std']:.3f}")
print(f"  LogLoss gap: {best['logloss_gap_mean']:+.3f}")

# Compare with previous best (nonconserved_residue notebook used Morgan 1024)
print("\n" + "=" * 90)
print("COMPARISON CONTEXT")
print("-" * 90)
print("Previous analyses:")
print("  - Nonconserved residue notebook: tested enzyme reprs with Morgan 1024 amine")
print("    -> Best: noncons_mean (unique, conservation < 0.5)")
print("  - Amine comparison notebook: tested amine reprs with noncons_mean enzyme")
print("    -> Best: F_physchem_onehot (ROC-AUC=0.797, PR-AUC=0.617)")
print(f"\nThis notebook: tested 8 enzyme reprs with F_physchem_onehot amine")
print(f"  -> Best: {best['representation']} (ROC-AUC={best['roc_auc_mean']:.3f}, PR-AUC={best['pr_auc_mean']:.3f})")
print("=" * 90)